# Tarefa 03

- Leia os enunciados com atenção
- Saiba que pode haver mais de uma resposta correta
- Insira novas células de código sempre que achar necessário
- Em caso de dúvidas, procure os Tutores
- Divirta-se :)

In [2]:
import pandas as pd
import requests

####  1) Lendo de APIs
Vimos em aula como carregar dados públicos do governo através de um API (*Application Programming Interface*). No exemplo de aula, baixamos os dados de pedidos de verificação de limites (PVL) realizados por estados, e selecionamos apenas aqueles referentes ao estado de São Paulo.

1. Repita os mesmos passos feitos em aula, mas selecione os PVLs realizados por municípios no estado do Rio de Janeiro.
2. Quais são os três *status* das solicitações mais frequentes na base? Quais são suas frequências?
3. Construa uma nova variável que contenha o ano do **status**. Observe que ```data_status``` vem como tipo *object* no **DataFrame**. Dica: você pode usar o método ```.str``` para transformar o tipo da variável em string, em seguida um método como [**slice()**](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.slice.html) ou [**split()**](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.split.html).
4. Indique a frequência de cada ano do campo construído no item (3).

In [3]:
# 1) Seu código aqui
url = 'https://apidatalake.tesouro.gov.br/ords/sadipem/tt/pvl'
r = requests.get(url)
r.raise_for_status() #verificar erro na requisição

data = r.json()

In [4]:
df = pd.DataFrame(data['items'])
# print(df.head())
# print(df.columns)
df_rj = df[df['uf'] == 'RJ']
print(df_rj.head())
    
    

     id_pleito tipo_interessado      interessado  cod_ibge  uf num_pvl  \
86       11576        Município           Quatis   3304128  RJ    None   
274      12505        Município    Volta Redonda   3306305  RJ    None   
278      11447        Município     Belford Roxo   3300456  RJ    None   
335      11343        Município  Duque de Caxias   3301702  RJ    None   
348      13080        Município        Três Rios   3306008  RJ    None   

           status          num_processo        data_protocolo  \
86   Regularizado  17944.001480/2009-15  2012-05-07T00:00:00Z   
274  Regularizado  17944.001701/2011-70  2013-12-04T00:00:00Z   
278      Deferido  17944.001450/2008-28  2008-06-10T00:00:00Z   
335  Regularizado  17944.001416/2006-91  2007-02-28T00:00:00Z   
348      Deferido  17944.001905/2014-53  2014-12-18T00:00:00Z   

                   tipo_operacao                                  finalidade  \
86   Operação contratual interna  Regularização de Dívida - Energia Elétrica   
274 

In [5]:
print(df.columns)

Index(['id_pleito', 'tipo_interessado', 'interessado', 'cod_ibge', 'uf',
       'num_pvl', 'status', 'num_processo', 'data_protocolo', 'tipo_operacao',
       'finalidade', 'tipo_credor', 'credor', 'moeda', 'valor',
       'pvl_assoc_divida', 'pvl_contradado_credor', 'data_status'],
      dtype='object')


In [6]:
# 2) Seu código aqui
# df_rj.columns
status_frequencia = df_rj['status'].value_counts()
top_3_status = status_frequencia.head(3)
#número total de solicitações
total_solicitacoes_rj = len(df_rj)
#porcentagem de cada status
status_percentual = (status_frequencia / total_solicitacoes_rj) * 100
top_3_status_percentual = (top_3_status / total_solicitacoes_rj) * 100

df_top_3_completo = pd.DataFrame({
    'Contagem': top_3_status,
    'Percentual (%)': top_3_status_percentual
})
print(df_top_3_completo)

                                                    Contagem  Percentual (%)
status                                                                      
Deferido                                                  29       37.179487
Arquivado                                                 15       19.230769
Encaminhado à PGFN com manifestação técnica fav...        12       15.384615


In [8]:
# 3) Seu código aqui
# checar se há valores nulos em 'data_'
print(f' valores nulos de data_status: {df_rj['data_status'].isnull().sum()}')

 valores nulos de data_status: 0


In [9]:

# criar uma cópia
df_rj_copy = df_rj.copy()

In [10]:
def extrair_ano_da_string(data_str):
    #se for nan ou não for str retorna none
    if pd.isna(data_str) or not isinstance(data_str, str):
        return None
    try:
        #ano ultimos 4 caracteres
        ano = data_str[-4:]
        return int(ano)
    except ValueError:
        print(f"Erro ao extrair ano da string: {data_str}")
        return None

In [25]:
# Garantir que a coluna 'data_status' seja do tipo string
df_rj_copy['data_status_str'] = df_rj_copy['data_status'].astype(str)

# # aplicar função para extrair o ano da string
df_rj_copy['data_status_ano'] = df_rj_copy['data_status_str'].apply(extrair_ano_da_string)

print("\nDataFrame com a nova coluna 'data_status' (extraída via string):")
df_rj_copy[['data_status', 'data_status_str', 'data_status_ano']].head()


DataFrame com a nova coluna 'data_status' (extraída via string):


,data_status,data_status_str,data_status_ano
86,29/06/2012,29/06/2012,2012
274,14/02/2014,14/02/2014,2014
278,10/06/2008,10/06/2008,2008
335,13/03/2007,13/03/2007,2007
348,07/01/2015,07/01/2015,2015


In [ ]:
# 4)
# frequência dos itens por ano na coluna criada no exercício 3
df_frequencia_ano = df_rj_copy['data_status_ano'].value_counts().reset_index()
df_frequencia_ano.columns = ['ano', 'frequencia']
df_frequencia_ano


,ano,frequencia
0,2007,12
1,2008,11
2,2012,6
3,2023,6
4,2014,6
5,2013,5
6,2011,4
7,2010,4
8,2009,3
9,2006,3


####  2) Melhorando a interação com o API
Observe dois URLs de consultas diferentes, por exemplo o URL utilizado em aula, e o URL feito no exercício anterior. Compare-os e observe as diferenças.

1. Faça uma função em Python que recebe como argumento o UF da consulta e o tipo de interessado (```'Estado'```ou ```Município```), e que devolve os dados da consulta no formato *DataFrame*.
2. Quantas solicitações para o Estado podem ser consultadas para Minas Gerais com *status* em 'Arquivado por decurso de prazo' estão registradas?
3. Qual é o município da Bahia com mais solicitações deferidas?
4. Salve um arquivo .csv com os dados de solicitações da Bahia, com interessado = 'Estado'

In [ ]:
#1) Seu código aqui

In [ ]:
# 2) Seu código aqui

In [ ]:
# 3) Seu código aqui

In [ ]:
# 4) Seu código aqui